# Job Listing Prioritization for Career Changers: Machine Learning

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

Builds and validates the prioritization model: which listings a career changer should apply to first, weighing whether they can realistically get each one (career-change accessibility, and later transferable fit) against whether it's worth applying now (timing, competition, pay). Listings a career changer can't get are already hidden; the model learns how much every other factor counts. It also powers **role discovery**: suggesting which role buckets (data and analytics, IT support, software development, …) a career changer's skills transfer to. Training profiles are the retrained career changers from `Profiles.ipynb` (plus volunteer profiles, if approved), aligned with the listings by role bucket. Uses the modeling-ready table from `EDA_SDA_4` and timing features from `TimeSeries`.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, and what's the scope? | Final table 1 |
| EDA_SDA_2 | Which listings do we study? Non-text checks, ghost listings, the career-change target | Niche table, non-text features |
| EDA_SDA_3 | What features does the text add? (NLP) | Niche table + text features |
| EDA_SDA_4 | Which features hold up? (statistics) | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing patterns and features |
| ML | How should listings be ranked for a career changer? | The prioritization model |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- How well can we predict whether a listing closes within N days, and how long it stays open?
- Which features drive the prediction most?
- Does the prioritization score beat how a career changer searches today (title keyword search, newest first)?
- Does adding career-change accessibility change a career changer's top 10, compared with timing factors alone?
- Does role discovery suggest the roles real career changers actually moved into (hit rate in the top 3 and top 5)?
- Does knowing what someone trained in predict their first tech role better than their background alone? (The evidence behind the retraining compass.)
- Does training the role model on everyone who reached tech (with origin as a feature) predict career changers' roles better than training on career changers alone? Results are always reported on career changers.
- **Role imbalance** (data and analytics is the smallest role): class weights first, per-role scores, and probabilities corrected to the real role mix. Synthetic people only for a role still predicted poorly, inside training folds only; never in the test set or any reported share.
- How many of a career changer's top 10 listings are still open when they apply, and realistically reachable?
- Is the model fast enough to rank listings live in the app?

**Data rules:** aggregates only. Clear outputs before committing.

### Roadmap and methods

| Section | Course standard (Intro2ML, SupervisedLearning, NumericalMethods, FinalProject, MLinBusiness) | Taken further |
|:--|:--|:--|
| 1. Setup | Load the modeling-ready table | |
| 2. Targets | Classification: closes within N days. Regression: days open | **Survival** target (days open with censoring) |
| 3. Preprocessing | Drop identifier columns; one-hot encoding with aligned columns; OrdinalEncoder for trees; **StandardScaler fit on training data only** | `Pipeline` + `ColumnTransformer`; rare levels grouped |
| 4. Split | Train / validation / test **once**, `random_state=12345`, test used once at the end | Split **by time** (train April–July, validate August, test September), folds **grouped by company**; folds also **grouped by applicant** (one person per fold, after duplicate removal), with each synthetic profile in the same fold as the real profile it came from |
| 5. Baselines | **DummyClassifier / DummyRegressor**, majority class, random | Newest-first ranking; title keyword search (how a career changer searches today) |
| 6. Class imbalance | `class_weight='balanced'`, upsampling (training only) | Decision-threshold tuning |
| 7. Models | **LogisticRegression, DecisionTree, RandomForest, LightGBM, CatBoost, XGBoost**, GradientBoosting, each tuned with loops or **GridSearchCV + cross-validation**; **`%%time`** for speed | **Cox regression** (with proportional-hazards checks), accelerated failure time models, survival forests, competing-risks models; **learning-to-rank** (LightGBM LambdaRank), which optimizes the order of an applicant's list directly |
| 8. Evaluation | ROC-AUC, F1, precision–recall and average precision, accuracy, **confusion matrix** (interpreted correctly), classification report; RMSE / MAE / R²; training vs. validation to check overfitting | **C-index** and time-dependent Brier score for survival; ranking metrics **precision@k** and **NDCG**; calibration; **conformal prediction** for apply-by ranges with guaranteed coverage; role discovery **hit rate@3 / @5** against the roles real switchers moved into |
| 9. Interpretation | **Feature importance** from the final model | **SHAP** values and partial dependence, for the app's "why it's ranked here" |
| 10. Business translation | Top-N selection by prediction; **bootstrap** confidence intervals and risk (MLinBusiness) | "Of a career changer's top 10, how many are still open and realistically reachable", with company-clustered intervals |
| 11. Sensitivity | | Do conclusions hold under our choices (2-day live tolerance, niche boundaries, CET `posted`)? |
| 12. Solution report | Steps performed, difficulties and how they were solved, the final model and its score | |

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
use_chart_style()

## 2.–12.

_To do, once EDA_SDA_4's table and the TimeSeries features exist._